# Clustering dos CPE com base nas features extraídas

Este notebook tem como objetivo agrupar os diferentes CPE com base nas features obtidas no notebook de exploração do dataset de energia.

Pretende-se:
- analisar a semelhança entre perfis de consumo;
- identificar grupos de CPE com comportamentos semelhantes;
- comparar diferentes métodos de clustering;
- persistir os modelos (scaler, PCA) para consistência entre notebooks;
- criar uma base interpretável para deteção de anomalias.

In [ ]:
# Imports e configuração

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
 
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score
from sklearn.ensemble import IsolationForest
from scipy.cluster.hierarchy import dendrogram, linkage
 
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")
 
results_dir = Path("../results")
models_dir = Path("../results/models")
results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Carregamento e inspeção das features

features = pd.read_csv(results_dir / "features_cpe.csv", index_col=0)
 
print("Shape:", features.shape)
print("Número de CPE:", features.index.nunique())
print("Número de features:", features.shape[1])
features.info()
features.describe()
 
print("\nNaN por coluna:")
display(features.isna().sum().sort_values(ascending=False))

# Observações

O dataset de entrada corresponde às features agregadas por CPE, construídas no notebook de exploração. Cada linha é um CPE, cada coluna uma característica do seu comportamento energético.

In [ ]:
# Normalização

X = features.copy()
feature_names = X.columns.tolist()
 
print("Features utilizadas no clustering:")
for col in feature_names:
    print(f"  - {col}")
 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
 
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
 
# Verificar normalização
print("\nVerificação da normalização:")
display(X_scaled_df.describe().T[["mean", "std"]].head(10))
 
# Persistir o scaler para reutilização nos notebooks seguintes
joblib.dump(scaler, models_dir / "scaler.pkl")
print("\nScaler guardado em:", models_dir / "scaler.pkl")

# Observações

As features apresentam escalas muito diferentes, pelo que a normalização com StandardScaler é necessária antes de qualquer método baseado em distância. O scaler é guardado para garantir consistência entre notebooks.

In [ ]:
# Deteção prévia de outliers

# Testar diferentes níveis de contaminação para entender a sensibilidade
contam_values = [0.01, 0.02, 0.03, 0.05]
outlier_results = {}
 
for contam in contam_values:
    iso = IsolationForest(contamination=contam, random_state=42)
    labels = iso.fit_predict(X_scaled)
    n_outliers = (labels == -1).sum()
    outlier_results[contam] = {
        "n_outliers": n_outliers,
        "pct": n_outliers / len(labels) * 100,
        "labels": labels
    }
    print(f"  contamination={contam:.2f} → {n_outliers} outliers ({n_outliers/len(labels)*100:.1f}%)")
 
# Escolha fundamentada: usar contamination que dá ~2-5% de outliers
CONTAM_ESCOLHIDA = 0.02
outlier_labels = outlier_results[CONTAM_ESCOLHIDA]["labels"]
 
print(f"\nContamination escolhida: {CONTAM_ESCOLHIDA}")
print(f"Outliers removidos: {outlier_results[CONTAM_ESCOLHIDA]['n_outliers']}")
 
# Separar normais e outliers
mask_normal = outlier_labels == 1
X_clean = X.loc[mask_normal].copy()
X_scaled_clean = X_scaled[mask_normal]
X_outliers = X.loc[~mask_normal].copy()
 
print(f"\nShape original: {X.shape}")
print(f"Shape sem outliers: {X_clean.shape}")
print(f"\nCPE identificados como outliers:")
display(X_outliers)

# Observações

Foi testada a sensibilidade do IsolationForest a diferentes valores de contamination. A escolha de contamination=0.02 é fundamentada pela necessidade de remover apenas pontos extremos sem perder informação relevante.

In [ ]:
# Redução de dimensionalidade (PCA)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled_clean)
 
df_pca = pd.DataFrame(X_pca, index=X_clean.index, columns=["PC1", "PC2"])
 
explained_variance = pca.explained_variance_ratio_
print(f"Variância explicada por componente:")
print(f"  PC1: {explained_variance[0]:.2%}")
print(f"  PC2: {explained_variance[1]:.2%}")
print(f"  Total: {explained_variance.sum():.2%}")
 
# Persistir o PCA para reutilização nos notebooks seguintes
joblib.dump(pca, models_dir / "pca.pkl")
print(f"\nPCA guardado em: {models_dir / 'pca.pkl'}")
 
# Visualização PCA sem clusters
plt.figure(figsize=(8, 6))
plt.scatter(df_pca["PC1"], df_pca["PC2"], alpha=0.7)
plt.title("Projeção dos CPE no espaço PCA (sem outliers)")
plt.xlabel(f"PC1 ({explained_variance[0]:.1%} variância)")
plt.ylabel(f"PC2 ({explained_variance[1]:.1%} variância)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
 
# Contribuição das features para cada componente
loadings = pd.DataFrame(
    pca.components_.T,
    columns=["PC1", "PC2"],
    index=feature_names
)
loadings["abs_PC1"] = loadings["PC1"].abs()
 
print("\nTop 5 features com maior peso no PC1:")
display(loadings.sort_values("abs_PC1", ascending=False).head(5)[["PC1", "PC2"]])

In [ ]:
# Escolha do número de clusters (K)

K_range = range(2, 11)
inertias = []
silhouette_scores_list = []
calinski_scores = []
davies_scores = []
 
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled_clean)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores_list.append(silhouette_score(X_scaled_clean, labels))
    calinski_scores.append(calinski_harabasz_score(X_scaled_clean, labels))
    davies_scores.append(davies_bouldin_score(X_scaled_clean, labels))
 
# Tabela resumo
df_k = pd.DataFrame({
    "K": list(K_range),
    "Inertia": inertias,
    "Silhouette": silhouette_scores_list,
    "Calinski-Harabasz": calinski_scores,
    "Davies-Bouldin": davies_scores
})
display(df_k)
 
# Gráficos
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
 
axes[0,0].plot(K_range, inertias, marker="o", color="#0D7C66")
axes[0,0].set_title("Método do Cotovelo (Elbow)")
axes[0,0].set_xlabel("K")
axes[0,0].set_ylabel("Inércia")
axes[0,0].set_xticks(list(K_range))
 
axes[0,1].plot(K_range, silhouette_scores_list, marker="o", color="#14A38B")
axes[0,1].set_title("Silhouette Score")
axes[0,1].set_xlabel("K")
axes[0,1].set_ylabel("Score")
axes[0,1].set_xticks(list(K_range))
 
axes[1,0].plot(K_range, calinski_scores, marker="o", color="#2980B9")
axes[1,0].set_title("Calinski-Harabasz (↑ melhor)")
axes[1,0].set_xlabel("K")
axes[1,0].set_ylabel("Score")
axes[1,0].set_xticks(list(K_range))
 
axes[1,1].plot(K_range, davies_scores, marker="o", color="#E74C3C")
axes[1,1].set_title("Davies-Bouldin (↓ melhor)")
axes[1,1].set_xlabel("K")
axes[1,1].set_ylabel("Score")
axes[1,1].set_xticks(list(K_range))
 
plt.suptitle("Seleção do número de clusters", fontweight="bold")
plt.tight_layout()
plt.savefig(results_dir / "selecao_k.png", dpi=150)
plt.show()
 
# Identificar melhor K por cada métrica
best_k_silhouette = df_k.loc[df_k["Silhouette"].idxmax(), "K"]
best_k_calinski = df_k.loc[df_k["Calinski-Harabasz"].idxmax(), "K"]
best_k_davies = df_k.loc[df_k["Davies-Bouldin"].idxmin(), "K"]
 
print(f"\nMelhor K por Silhouette: {best_k_silhouette}")
print(f"Melhor K por Calinski-Harabasz: {best_k_calinski}")
print(f"Melhor K por Davies-Bouldin: {best_k_davies}")
 
# Definir K com base na análise
K_opt = 3

# Observações

A escolha do K é fundamentada em 4 métricas:
- Elbow: ponto de inflexão na curva de inércia;
- Silhouette: qualidade da separação entre clusters (máximo);
- Calinski-Harabasz: rácio entre dispersão inter/intra-cluster (máximo);
- Davies-Bouldin: semelhança média entre clusters (mínimo).

In [ ]:
# Comparação de métodos de clustering

print("COMPARAÇÃO DE MÉTODOS DE CLUSTERING")
 
resultados_metodos = {}
 
# KMeans
kmeans = KMeans(n_clusters=K_opt, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_scaled_clean)
 
resultados_metodos["KMeans"] = {
    "labels": labels_kmeans,
    "silhouette": silhouette_score(X_scaled_clean, labels_kmeans),
    "calinski": calinski_harabasz_score(X_scaled_clean, labels_kmeans),
    "davies": davies_bouldin_score(X_scaled_clean, labels_kmeans),
    "n_clusters": len(set(labels_kmeans)),
}
 
# Agglomerative Clustering
agglo = AgglomerativeClustering(n_clusters=K_opt)
labels_agglo = agglo.fit_predict(X_scaled_clean)
 
resultados_metodos["Agglomerative"] = {
    "labels": labels_agglo,
    "silhouette": silhouette_score(X_scaled_clean, labels_agglo),
    "calinski": calinski_harabasz_score(X_scaled_clean, labels_agglo),
    "davies": davies_bouldin_score(X_scaled_clean, labels_agglo),
    "n_clusters": len(set(labels_agglo)),
}
 
# Gaussian Mixture Models
gmm = GaussianMixture(n_components=K_opt, random_state=42, covariance_type="full")
labels_gmm = gmm.fit_predict(X_scaled_clean)
 
resultados_metodos["GMM"] = {
    "labels": labels_gmm,
    "silhouette": silhouette_score(X_scaled_clean, labels_gmm),
    "calinski": calinski_harabasz_score(X_scaled_clean, labels_gmm),
    "davies": davies_bouldin_score(X_scaled_clean, labels_gmm),
    "n_clusters": len(set(labels_gmm)),
}
 
# DBSCAN
# Testar diferentes eps
from sklearn.neighbors import NearestNeighbors
 
# Calcular distância ao k-ésimo vizinho mais próximo para estimar eps
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled_clean)
distances, _ = nn.kneighbors(X_scaled_clean)
k_distances = np.sort(distances[:, -1])
 
plt.figure(figsize=(8, 4))
plt.plot(k_distances)
plt.title("K-distance plot (k=5) para estimação de eps")
plt.xlabel("Pontos ordenados")
plt.ylabel("Distância ao 5º vizinho")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
 
# Testar vários eps
eps_values = [1.5, 2.0, 2.5, 3.0, 3.5]
dbscan_results = []
 
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=3)
    labels_db = db.fit_predict(X_scaled_clean)
    n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
    n_noise = (labels_db == -1).sum()
    
    sil = silhouette_score(X_scaled_clean, labels_db) if n_clusters >= 2 else -1
    
    dbscan_results.append({
        "eps": eps,
        "n_clusters": n_clusters,
        "n_noise": n_noise,
        "silhouette": sil
    })
    print(f"  eps={eps:.1f} → {n_clusters} clusters, {n_noise} noise points, silhouette={sil:.3f}")
 
# Escolher melhor eps (com pelo menos 2 clusters e melhor silhouette)
df_dbscan = pd.DataFrame(dbscan_results)
valid_dbscan = df_dbscan[df_dbscan["n_clusters"] >= 2]
 
if not valid_dbscan.empty:
    best_eps = valid_dbscan.loc[valid_dbscan["silhouette"].idxmax(), "eps"]
    db_final = DBSCAN(eps=best_eps, min_samples=3)
    labels_dbscan = db_final.fit_predict(X_scaled_clean)
    n_clusters_db = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    
    if n_clusters_db >= 2:
        resultados_metodos["DBSCAN"] = {
            "labels": labels_dbscan,
            "silhouette": silhouette_score(X_scaled_clean, labels_dbscan),
            "calinski": calinski_harabasz_score(X_scaled_clean[labels_dbscan != -1], labels_dbscan[labels_dbscan != -1]) if (labels_dbscan != -1).sum() > n_clusters_db else 0,
            "davies": davies_bouldin_score(X_scaled_clean[labels_dbscan != -1], labels_dbscan[labels_dbscan != -1]) if (labels_dbscan != -1).sum() > n_clusters_db else 999,
            "n_clusters": n_clusters_db,
        }
        print(f"\n  DBSCAN melhor eps: {best_eps}")
 
# Tabela comparativa
df_comparacao = pd.DataFrame({
    method: {
        "Clusters": res["n_clusters"],
        "Silhouette ↑": round(res["silhouette"], 3),
        "Calinski-Harabasz ↑": round(res["calinski"], 1),
        "Davies-Bouldin ↓": round(res["davies"], 3),
    }
    for method, res in resultados_metodos.items()
}).T
 
print("\n" + "="*60)
print("COMPARAÇÃO DE MÉTODOS")
print("="*60)
display(df_comparacao)
 
# Guardar tabela
df_comparacao.to_csv(results_dir / "comparacao_clustering.csv")
 
# Visualização PCA por método
fig, axes = plt.subplots(1, len(resultados_metodos), figsize=(5*len(resultados_metodos), 5))
if len(resultados_metodos) == 1:
    axes = [axes]
 
for ax, (method, res) in zip(axes, resultados_metodos.items()):
    scatter = ax.scatter(
        df_pca["PC1"], df_pca["PC2"],
        c=res["labels"], cmap="tab10", alpha=0.7, s=40
    )
    ax.set_title(f"{method}\nSilhouette: {res['silhouette']:.3f}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.3)
 
plt.suptitle("Comparação de métodos de clustering (projeção PCA)", fontweight="bold")
plt.tight_layout()
plt.savefig(results_dir / "comparacao_clustering_pca.png", dpi=150)
plt.show()

# Observações

Foram comparados 4 métodos de clustering:
- KMeans: clusters esféricos, sensível a outliers
- Agglomerative: hierárquico, sem assunção de forma
- GMM: clusters elipsoidais, modela a incerteza
- DBSCAN: baseado em densidade, deteta ruído

A comparação permite avaliar a estabilidade dos agrupamentos e selecionar o método mais adequado para os dados.

In [ ]:
# Análise final dos clusters (KMeans)

K_opt = 3

kmeans = KMeans(n_clusters=K_opt, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled_clean)

df_pca["cluster"] = clusters

print(f"K = {K_opt}")
print(f"Número de CPE por cluster:")
print(pd.Series(clusters).value_counts().sort_index())

In [ ]:
# Método selecionado: análise detalhada

# Selecionar o melhor método
METODO_ESCOLHIDO = "KMeans"
print(f"Método escolhido: {METODO_ESCOLHIDO}")
 
clusters_labels = resultados_metodos[METODO_ESCOLHIDO]["labels"]
 
df_clusters = X_clean.copy()
df_clusters["cluster"] = clusters_labels
 
print(f"\nNúmero de CPE por cluster:")
display(df_clusters["cluster"].value_counts().sort_index())
 
# Visualização PCA com centróides
df_pca["cluster"] = clusters_labels
 
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    df_pca["PC1"], df_pca["PC2"],
    c=df_pca["cluster"], cmap="tab10", alpha=0.7, s=50
)
 
if METODO_ESCOLHIDO == "KMeans":
    centroids_pca = pca.transform(kmeans.cluster_centers_)
    plt.scatter(
        centroids_pca[:, 0], centroids_pca[:, 1],
        c="black", s=200, marker="X", label="Centróides", zorder=5
    )
    plt.legend()
 
plt.title(f"Clusters de consumo dos CPE ({METODO_ESCOLHIDO}, projeção PCA)")
plt.xlabel(f"PC1 ({explained_variance[0]:.1%} variância)")
plt.ylabel(f"PC2 ({explained_variance[1]:.1%} variância)")
plt.colorbar(scatter, label="Cluster")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(results_dir / "clusters_pca.png", dpi=150)
plt.show()
 
# Análise de silhouette por CPE
sample_silhouette = silhouette_samples(X_scaled_clean, clusters_labels)
 
fig, ax = plt.subplots(figsize=(8, 6))
y_lower = 10
 
for i in sorted(df_clusters["cluster"].unique()):
    cluster_silhouette = sample_silhouette[clusters_labels == i]
    cluster_silhouette.sort()
    
    size = cluster_silhouette.shape[0]
    y_upper = y_lower + size
    
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size, str(i))
    y_lower = y_upper + 10
 
ax.axvline(x=silhouette_score(X_scaled_clean, clusters_labels), color="red", linestyle="--", label="Média")
ax.set_title("Silhouette por CPE e cluster")
ax.set_xlabel("Silhouette Score")
ax.set_ylabel("Cluster")
ax.legend()
plt.tight_layout()
plt.savefig(results_dir / "silhouette_por_cpe.png", dpi=150)
plt.show()

In [ ]:
# 8. Interpretação dos clusters

# Perfil médio das features por cluster
cluster_summary = df_clusters.groupby("cluster").mean()
 
print("\nPerfil médio das features por cluster:")
display(cluster_summary.T.round(2))
 
# Heatmap normalizado
cluster_summary_norm = (cluster_summary - cluster_summary.mean()) / cluster_summary.std()
 
plt.figure(figsize=(14, 6))
sns.heatmap(
    cluster_summary_norm.T, 
    annot=True, fmt=".1f", cmap="RdBu_r", center=0,
    linewidths=0.5
)
plt.title("Perfil normalizado das features por cluster (z-score relativo)")
plt.xlabel("Cluster")
plt.ylabel("Features")
plt.tight_layout()
plt.savefig(results_dir / "heatmap_clusters.png", dpi=150)
plt.show()
 
# Gerar interpretação automática
print("\n" + "="*60)
print("INTERPRETAÇÃO DOS CLUSTERS")
print("="*60)
 
for cluster_id in sorted(df_clusters["cluster"].unique()):
    n_cpes = (df_clusters["cluster"] == cluster_id).sum()
    row = cluster_summary.loc[cluster_id]
    
    print(f"\nCluster {cluster_id} ({n_cpes} CPEs)")
    print(f"  Consumo médio: {row['mean']:.2f}")
    print(f"  Coef. variação: {row['cv']:.2f}")
    
    if "ratio_noite_dia" in row:
        if row["ratio_noite_dia"] > 1:
            print(f"  Perfil: consumo predominantemente NOTURNO (ratio={row['ratio_noite_dia']:.2f})")
        else:
            print(f"  Perfil: consumo predominantemente DIURNO (ratio={row['ratio_noite_dia']:.2f})")
    
    if "autocorr_lag96" in row:
        if row["autocorr_lag96"] > 0.7:
            print(f"  Regularidade diária: ALTA (autocorr_lag96={row['autocorr_lag96']:.2f})")
        elif row["autocorr_lag96"] > 0.4:
            print(f"  Regularidade diária: MODERADA (autocorr_lag96={row['autocorr_lag96']:.2f})")
        else:
            print(f"  Regularidade diária: BAIXA (autocorr_lag96={row['autocorr_lag96']:.2f})")
 
# Boxplots das features mais discriminativas
features_to_plot = ["mean", "cv", "ratio_noite_dia", "amplitude_horaria", "diff_weekday_weekend"]
features_to_plot = [f for f in features_to_plot if f in df_clusters.columns]
 
fig, axes = plt.subplots(1, len(features_to_plot), figsize=(4*len(features_to_plot), 4))
 
for ax, feat in zip(axes, features_to_plot):
    sns.boxplot(data=df_clusters, x="cluster", y=feat, ax=ax, palette="tab10")
    ax.set_title(feat)
 
plt.suptitle("Distribuição de features por cluster", fontweight="bold")
plt.tight_layout()
plt.savefig(results_dir / "boxplots_clusters.png", dpi=150)
plt.show()

In [ ]:
# Consolidação e exportação

# Juntar outliers
df_outlier_result = X_outliers.copy()
df_outlier_result["cluster"] = "outlier"
 
df_clusters_final = pd.concat([df_clusters, df_outlier_result])
 
print(f"\nDistribuição final:")
display(df_clusters_final["cluster"].value_counts())
 
# Guardar tudo
df_clusters_final.to_csv(results_dir / "clusters_cpe.csv")
print("Clusters guardados:", results_dir / "clusters_cpe.csv")
 
df_pca.to_csv(results_dir / "pca_clusters_cpe.csv")
print("Projeção PCA guardada:", results_dir / "pca_clusters_cpe.csv")
 
cluster_summary.to_csv(results_dir / "cluster_summary.csv")
print("Resumo dos clusters guardado:", results_dir / "cluster_summary.csv")
 
# Guardar o modelo KMeans e a mask de outliers
if METODO_ESCOLHIDO == "KMeans":
    joblib.dump(kmeans, models_dir / "kmeans.pkl")
    print("Modelo KMeans guardado:", models_dir / "kmeans.pkl")
 
# Guardar a mask de normais (para o notebook de anomalias saber quais CPEs excluir)
pd.Series(mask_normal, index=X.index, name="is_normal").to_csv(results_dir / "outlier_mask.csv")
print("Máscara de outliers guardada:", results_dir / "outlier_mask.csv")
 
# Listar modelos persistidos
print("\n" + "="*60)
print("MODELOS PERSISTIDOS (para reutilização)")
print("="*60)
print(f"  {models_dir / 'scaler.pkl'}")
print(f"  {models_dir / 'pca.pkl'}")
if METODO_ESCOLHIDO == "KMeans":
    print(f"  {models_dir / 'kmeans.pkl'}")

# Conclusões

A aplicação de clustering permitiu identificar grupos de CPE com comportamentos energéticos distintos.

Foram comparados 4 métodos (KMeans, Agglomerative, GMM, DBSCAN),com métricas quantitativas para fundamentar a escolha.

Os modelos (scaler, PCA) foram persistidos para garantir consistência nas visualizações dos notebooks seguintes.

Próximos passos:
- Deteção de anomalias ao nível das features (por cluster)
- Deteção de anomalias temporais
- Ambos usando os mesmos modelos persistidos aqui